# Installation

In [ ]:
!pip install -q llmcompressor datasets transformers accelerate

In [1]:
import os

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN is None:
    print("Token not found!")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded successfully.")

Token not found!


In [2]:
# -*- coding: utf-8 -*-

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
import pickle
from tqdm import tqdm

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

SEED = 42
set_seed(SEED)

print("=" * 70)
print(f"AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED 4-BIT, DYNAMIC λ, QKV-ONLY + FP32 Wo, GPT2+LLaMA)")
print(f"SEED: {SEED}")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

class STEQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, scale, q_max):
        scale_c = torch.clamp(scale, min=1e-8)
        w_scaled = weight / scale_c
        w_clamped = torch.clamp(w_scaled, -q_max, q_max)
        w_round = torch.round(w_clamped)
        w_quant = w_round * scale_c

        ctx.save_for_backward(w_scaled, w_clamped, w_round, scale_c)
        ctx.q_max = q_max
        return w_quant

    @staticmethod
    def backward(ctx, grad_output):
        w_scaled, w_clamped, w_round, scale_c = ctx.saved_tensors
        q_max = ctx.q_max

        not_clipped = (w_scaled.abs() <= q_max).float()
        grad_weight = grad_output * not_clipped

        grad_scale_elem = torch.where(
            w_scaled > q_max, torch.full_like(w_scaled, float(q_max)),
            torch.where(
                w_scaled < -q_max, torch.full_like(w_scaled, -float(q_max)),
                w_round - w_scaled
            )
        )
        contrib = grad_output * grad_scale_elem
        grad_scale = contrib.sum(dim=0, keepdim=True)

        return grad_weight, grad_scale, None

class QuantizedLinear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor = None, bits: int = 4):
        super().__init__()
        self.register_buffer("weight_fp", weight.clone().detach())
        if bias is not None:
            self.register_buffer("bias_fp", bias.clone().detach())
        else:
            self.bias_fp = None

        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1

        init_scale = (
            weight.detach().abs().amax(dim=0, keepdim=True) / self.q_max
        ).clamp(min=1e-6)
        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale = nn.Parameter(init_raw)

    @property
    def scale(self):
        return F.softplus(self.raw_scale) + 1e-8

    def forward(self, x: torch.Tensor, force_fp: bool = False) -> torch.Tensor:
        if force_fp:
            w = self.weight_fp
        else:
            w = STEQuantize.apply(self.weight_fp, self.scale, self.q_max)

        out = torch.matmul(x, w)
        if self.bias_fp is not None:
            out = out + self.bias_fp
        return out
        
    def set_bits(self, bits: int):
        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1

        init_scale = (
            self.weight_fp.abs().amax(dim=0, keepdim=True) / self.q_max
        ).clamp(min=1e-6)

        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale.data = init_raw

class QuantizedGPT2Attention(nn.Module):
    def __init__(self, original_attn: nn.Module, bits: int = 4):
        super().__init__()

        self.force_fp_mode = False

        self.original_attn = original_attn
        self.embed_dim = original_attn.embed_dim
        self.num_heads = original_attn.num_heads
        self.head_dim = self.embed_dim // self.num_heads
        self.attn_dropout = original_attn.attn_dropout
        self.resid_dropout = original_attn.resid_dropout

        W = original_attn.c_attn.weight.data
        b = original_attn.c_attn.bias.data if original_attn.c_attn.bias is not None else None

        w_q, w_k, w_v = W.chunk(3, dim=-1)
        b_q, b_k, b_v = b.chunk(3, dim=-1) if b is not None else (None, None, None)

        self.q_proj = QuantizedLinear(w_q, b_q, bits=bits)
        self.k_proj = QuantizedLinear(w_k, b_k, bits=bits)
        self.v_proj = QuantizedLinear(w_v, b_v, bits=bits)

        self.o_proj_fp = original_attn.c_proj

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, H, T, D = x.shape
        return x.permute(0, 2, 1, 3).contiguous().view(B, T, self.embed_dim)

    def forward_components(self, hidden_states: torch.Tensor, use_quant: bool = True):
        B, T, _ = hidden_states.shape

        if use_quant:
            q = self.q_proj(hidden_states)
            k = self.k_proj(hidden_states)
            v = self.v_proj(hidden_states)
        else:
            q = self.q_proj(hidden_states, force_fp=True)
            k = self.k_proj(hidden_states, force_fp=True)
            v = self.v_proj(hidden_states, force_fp=True)

        q = self._split_heads(q)
        k = self._split_heads(k)
        v = self._split_heads(v)

        scale = 1.0 / math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-1, -2)) * scale

        causal_mask = torch.triu(
            torch.ones((T, T), device=scores.device, dtype=torch.bool),
            diagonal=1
        )

        mask_value = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), mask_value)

        P = F.softmax(scores, dim=-1, dtype=torch.float32).to(scores.dtype)
        P = self.attn_dropout(P)

        O = torch.matmul(P, v)
        O_merged = self._merge_heads(O)

        return P, O_merged, causal_mask

    def forward(self, hidden_states, layer_past=None, past_key_values=None,
                attention_mask=None, head_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, use_cache=False, output_attentions=False,
                **kwargs):
        if layer_past is None:
            layer_past = past_key_values
        if layer_past is not None:
            raise NotImplementedError("KV-cache not supported.")

        use_quant = not self.force_fp_mode
        P, O_merged, _ = self.forward_components(hidden_states, use_quant=use_quant)

        attn_output = self.o_proj_fp(O_merged)
        attn_output = self.resid_dropout(attn_output)

        outputs = (attn_output,)
        if use_cache:
            outputs = outputs + (None,)
        if output_attentions:
            outputs = outputs + (P,)
        if len(outputs) == 1:
            outputs = outputs + (None,)
        return outputs

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def repeat_kv(hidden_states, n_rep):
    batch, num_kv_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, slen, head_dim)

class LlamaRotaryHelper(nn.Module):
    def __init__(self, head_dim, config=None, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))

        rope_scaling = getattr(config, "rope_scaling", None) if config is not None else None
        if rope_scaling is not None and rope_scaling.get("rope_type") == "llama3":
            factor = rope_scaling["factor"]
            low_freq_factor = rope_scaling["low_freq_factor"]
            high_freq_factor = rope_scaling["high_freq_factor"]
            old_context_len = rope_scaling["original_max_position_embeddings"]

            low_freq_wavelen = old_context_len / low_freq_factor
            high_freq_wavelen = old_context_len / high_freq_factor
            wavelen = 2 * math.pi / inv_freq

            inv_freq_llama = torch.where(wavelen > low_freq_wavelen, inv_freq / factor, inv_freq)
            smooth_factor = (old_context_len / wavelen - low_freq_factor) / (high_freq_factor - low_freq_factor)
            smoothed_inv_freq = smooth_factor * inv_freq_llama / factor + (1 - smooth_factor) * inv_freq_llama
            is_medium_freq = ~(wavelen < high_freq_wavelen) & ~(wavelen > low_freq_wavelen)
            inv_freq = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)

        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x, position_ids):
        inv_freq_expanded = self.inv_freq[None, :, None].float().to(x.device)
        inv_freq_expanded = inv_freq_expanded.expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(dtype=x.dtype), emb.sin().to(dtype=x.dtype)

class QuantizedLlamaAttention(nn.Module):
    def __init__(self, original_attn, bits=4, rotary_emb=None):
        super().__init__()

        self.force_fp_mode = False

        self.original_attn = original_attn
        cfg = original_attn.config
        self.rotary_emb = rotary_emb

        self.embed_dim = cfg.hidden_size
        self.num_heads = cfg.num_attention_heads
        self.num_kv_heads = getattr(cfg, "num_key_value_heads", self.num_heads)
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.head_dim = getattr(original_attn, "head_dim", self.embed_dim // self.num_heads)
        self.rope_theta = getattr(cfg, "rope_theta", 10000.0)
        self.attn_dropout = nn.Dropout(getattr(original_attn, "attention_dropout", 0.0))

        W_q = original_attn.q_proj.weight.data.T.contiguous()
        b_q = original_attn.q_proj.bias.data if original_attn.q_proj.bias is not None else None
        W_k = original_attn.k_proj.weight.data.T.contiguous()
        b_k = original_attn.k_proj.bias.data if original_attn.k_proj.bias is not None else None
        W_v = original_attn.v_proj.weight.data.T.contiguous()
        b_v = original_attn.v_proj.bias.data if original_attn.v_proj.bias is not None else None

        self.q_proj = QuantizedLinear(W_q, b_q, bits=bits)
        self.k_proj = QuantizedLinear(W_k, b_k, bits=bits)
        self.v_proj = QuantizedLinear(W_v, b_v, bits=bits)

        self.o_proj_fp = original_attn.o_proj

        self.rotary = LlamaRotaryHelper(self.head_dim, config=cfg, base=self.rope_theta)

    def _split_heads(self, x, num_heads):
        B, T, _ = x.shape
        return x.view(B, T, num_heads, self.head_dim).permute(0, 2, 1, 3)

    def _merge_heads(self, x):
        B, H, T, D = x.shape
        return x.permute(0, 2, 1, 3).contiguous().view(B, T, H * D)

    def forward_components(self, hidden_states, use_quant=True, position_embeddings=None):
        B, T, _ = hidden_states.shape

        if use_quant:
            q = self.q_proj(hidden_states)
            k = self.k_proj(hidden_states)
            v = self.v_proj(hidden_states)
        else:
            q = self.q_proj(hidden_states, force_fp=True)
            k = self.k_proj(hidden_states, force_fp=True)
            v = self.v_proj(hidden_states, force_fp=True)

        q = self._split_heads(q, self.num_heads)
        k = self._split_heads(k, self.num_kv_heads)
        v = self._split_heads(v, self.num_kv_heads)

        if position_embeddings is not None:
            cos, sin = position_embeddings
        elif self.rotary_emb is not None:
            position_ids = torch.arange(T, device=hidden_states.device).unsqueeze(0).expand(B, -1)
            cos, sin = self.rotary_emb(hidden_states, position_ids)
        else:
            raise RuntimeError("No RoPE available")

        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        k = repeat_kv(k, self.num_kv_groups)
        v = repeat_kv(v, self.num_kv_groups)

        scale = 1.0 / math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-1, -2)) * scale

        causal_mask = torch.triu(
            torch.ones((T, T), device=scores.device, dtype=torch.bool), diagonal=1
        )

        mask_value = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), mask_value)

        P = F.softmax(scores, dim=-1, dtype=torch.float32).to(scores.dtype)
        P = self.attn_dropout(P)

        O = torch.matmul(P, v)
        O_merged = self._merge_heads(O)
        return P, O_merged, causal_mask

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                past_key_value=None, output_attentions=False,
                position_embeddings=None, **kwargs):
        use_quant = not self.force_fp_mode
        P, O_merged, _ = self.forward_components(
            hidden_states,
            use_quant=use_quant,
            position_embeddings=position_embeddings,
        )
        
        attn_output = self.o_proj_fp(O_merged)
        outputs = (attn_output, None)
        if output_attentions:
            outputs = outputs + (P,)
        return outputs

def compute_ap_loss_improved(P_fp, O_fp, P_q, O_q, causal_mask,
                         lam=1.0, temp=2.0, eps=1e-8):
    diff = O_fp - O_q
    mse_raw = diff.pow(2).mean()

    l_output = mse_raw / (O_fp.pow(2).mean() + eps)

    P_fp_c = F.softmax(torch.log(P_fp.clamp(min=eps)) / temp, dim=-1)
    P_q_c = F.softmax(torch.log(P_q.clamp(min=eps)) / temp, dim=-1)

    kl_matrix = P_fp_c * (torch.log(P_fp_c + eps) - torch.log(P_q_c + eps))

    valid_mask = (~causal_mask).unsqueeze(0).unsqueeze(0)
    kl_valid = kl_matrix.masked_fill(~valid_mask, 0.0)

    kl_per_query = kl_valid.sum(dim=-1)
    query_valid = valid_mask.any(dim=-1).to(kl_per_query.dtype)
    l_kl = (kl_per_query * query_valid).sum() / query_valid.sum().clamp_min(1.0)

    l_joint = l_output + lam * l_kl
    return l_joint, l_output, l_kl, mse_raw

def detect_arch(model):
    if hasattr(model, "transformer"):
        return "gpt2"
    if hasattr(model, "model"):
        return "llama"
    raise ValueError("Unsupported model architecture")

def quantize_transformer_attn(model, layer_bits=None):
    arch = detect_arch(model)
    q_attn_blocks = []

    if arch == "gpt2":
        for layer_idx, block in enumerate(model.transformer.h):
            bits = layer_bits[layer_idx] if layer_bits is not None else 4
            q_attn = QuantizedGPT2Attention(block.attn, bits=bits).to(device)
            block.attn = q_attn
            q_attn_blocks.append(q_attn)
            print(f"  GPT2 layer {layer_idx:02d}: attention quantized to {bits}-bit (QKV-ONLY + FP32 Wo)")
    elif arch == "llama":
        shared_rotary_emb = model.model.rotary_emb
        for layer_idx, block in enumerate(model.model.layers):
            bits = layer_bits[layer_idx] if layer_bits is not None else 4
            q_attn = QuantizedLlamaAttention(
                block.self_attn,
                bits=bits,
                rotary_emb=shared_rotary_emb,
            ).to(device)
            block.self_attn = q_attn
            q_attn_blocks.append(q_attn)
            print(f"  LLaMA layer {layer_idx:02d}: attention quantized to {bits}-bit (QKV-ONLY + FP32 Wo)")
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    return arch, q_attn_blocks

def collect_layer_inputs(model, arch: str, calib_tokens, device):
    if arch == "gpt2":
        num_layers = len(model.transformer.h)
    elif arch == "llama":
        num_layers = len(model.model.layers)
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    layer_inputs = [[] for _ in range(num_layers)]

    def make_hook(layer_idx):
        def hook_fn(module, inp, out):
            layer_inputs[layer_idx].append(out.detach())
        return hook_fn

    handles = []
    if arch == "gpt2":
        for layer_idx, block in enumerate(model.transformer.h):
            h = block.ln_1.register_forward_hook(make_hook(layer_idx))
            handles.append(h)
    else:
        for layer_idx, block in enumerate(model.model.layers):
            h = block.input_layernorm.register_forward_hook(make_hook(layer_idx))
            handles.append(h)

    with torch.no_grad():
        for chunk in calib_tokens:
            model(chunk.to(device), use_cache=False)

    for h in handles:
        h.remove()

    return layer_inputs

@torch.no_grad()
def compute_sensitivity_matrix(
    model,
    calib_tokens,
    candidate_bits,
    temp: float = 2.0,
    device: str = "cuda",
    lambda_samples: int = 8,
):
    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=None)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    layer_inputs = collect_layer_inputs(model, arch, calib_tokens, device)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    lam_values = []
    valid_layers = [i for i in range(len(q_attn_blocks)) if len(layer_inputs[i]) > 0]
    if not valid_layers:
        raise RuntimeError("No layer inputs collected; cannot estimate lambda.")

    num_layers_to_use = min(3, len(valid_layers))
    layers_to_use = valid_layers[:num_layers_to_use]

    for layer_idx in layers_to_use:
        q_attn = q_attn_blocks[layer_idx]

        q_attn.q_proj.set_bits(4)
        q_attn.k_proj.set_bits(4)
        q_attn.v_proj.set_bits(4)

        n_samples = min(lambda_samples, len(layer_inputs[layer_idx]))
        sample_indices = np.random.choice(len(layer_inputs[layer_idx]), n_samples, replace=False)

        for idx in sample_indices:
            inp = layer_inputs[layer_idx][idx].to(device)

            q_attn.force_fp_mode = True
            P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)

            q_attn.force_fp_mode = False
            P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)

            _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                P_fp, O_fp, P_q, O_q, mask,
                lam=1.0,
                temp=temp,
            )

            lam_values.append(float(l_out_0 / (l_kl_0 + 1e-8)))

    lam = np.mean(lam_values) if lam_values else 1.0
    lam = min(max(lam, 0.01), 10.0)
    print(f"[Sensitivity] λ = {lam:.4f}")

    sensitivity = {i: {} for i in range(len(q_attn_blocks))}

    for layer_idx, q_attn in enumerate(q_attn_blocks):
        if len(layer_inputs[layer_idx]) == 0:
            continue

        print(f"layer {layer_idx:02d}:", end=" ")

        for bits in candidate_bits:
            q_attn.q_proj.set_bits(bits)
            q_attn.k_proj.set_bits(bits)
            q_attn.v_proj.set_bits(bits)

            total_l_joint = 0.0
            n_samples = len(layer_inputs[layer_idx])

            for inp in layer_inputs[layer_idx]:
                inp = inp.to(device)

                q_attn.force_fp_mode = True
                P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)

                q_attn.force_fp_mode = False
                P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)

                l_joint, _, _, _ = compute_ap_loss_improved(
                    P_fp, O_fp, P_q, O_q, mask,
                    lam=lam,
                    temp=temp,
                )
                total_l_joint += l_joint.item()

            avg_loss = total_l_joint / max(n_samples, 1)
            sensitivity[layer_idx][bits] = avg_loss

            print(f"{bits}b={avg_loss:.4f}", end=", ")

        print()

    return sensitivity, lam

def solve_mckp(sensitivity, cost, budget, candidate_bits):
    n = len(sensitivity)
    INF = float("inf")
    dp = [INF] * (budget + 1)
    dp[0] = 0.0
    choice = [[None] * (budget + 1) for _ in range(n)]

    for i in range(n):
        new_dp = [INF] * (budget + 1)
        for c in range(budget + 1):
            if dp[c] == INF:
                continue
            for b in candidate_bits:
                c2 = c + cost[b]
                if c2 <= budget and dp[c] + sensitivity[i][b] < new_dp[c2]:
                    new_dp[c2] = dp[c] + sensitivity[i][b]
                    choice[i][c2] = (b, c)
        dp = new_dp

    best_c = min(range(budget + 1), key=lambda c: dp[c])
    assignment, c = {}, best_c
    for i in reversed(range(n)):
        b, prev_c = choice[i][c]
        assignment[i] = b
        c = prev_c
    return assignment, dp[best_c]

def calibrate_ap_quant_sequential(
    model,
    calib_tokens,
    device,
    layer_bits=None,
    num_steps_per_layer=200,
    lr=1e-3,
    init_temp=2.0,
    lambda_update_every=10,
    batch_size=4,
    val_fraction=0.1,
    calibration_passes=2,
):
    model.eval()
    model.to(device)

    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=layer_bits)

    print("\n" + "=" * 70)
    print(f"AP-QUANT: SEQUENTIAL CALIBRATION ({arch.upper()}, 4-BIT, DYNAMIC λ, QKV-ONLY + FP32 Wo)")
    print(f"Steps/layer: {num_steps_per_layer}, Batch size: {batch_size}, Passes: {calibration_passes}")
    print(f"Full validation set checkpointing: ENABLED")
    print("=" * 70)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    layer_inputs_fp = collect_layer_inputs(model, arch=arch, calib_tokens=calib_tokens, device=device)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    print("\n  Initializing scales with RTN...")
    for layer_idx, q_attn in enumerate(q_attn_blocks):
        if layer_bits is not None and layer_idx in layer_bits:
            bits = layer_bits[layer_idx]
            q_attn.q_proj.set_bits(bits)
            q_attn.k_proj.set_bits(bits)
            q_attn.v_proj.set_bits(bits)
        else:
            q_attn.q_proj.set_bits(4)
            q_attn.k_proj.set_bits(4)
            q_attn.v_proj.set_bits(4)
        
        q_attn.force_fp_mode = False

    best_scales_overall = [(
        q_attn.q_proj.raw_scale.data.clone(),
        q_attn.k_proj.raw_scale.data.clone(),
        q_attn.v_proj.raw_scale.data.clone(),
    ) for q_attn in q_attn_blocks]
    
    best_overall_loss = float('inf')

    for pass_idx in range(calibration_passes):
        print(f"\n{'='*70}")
        print(f"SEQUENTIAL CALIBRATION PASS {pass_idx + 1}/{calibration_passes}")
        print(f"{'='*70}")

        for layer_idx, q_attn in enumerate(q_attn_blocks):
            if len(layer_inputs_fp[layer_idx]) == 0:
                print(f"  Layer {layer_idx:02d}: no inputs collected, skipping.")
                continue

            print(f"\n--- Calibrating layer {layer_idx:02d} (pass {pass_idx + 1}) ---")

            q_attn.force_fp_mode = True
            layer_inputs_quant = collect_layer_inputs(
                model, arch=arch, calib_tokens=calib_tokens, device=device
            )
            q_attn.force_fp_mode = False

            current_inputs = layer_inputs_quant[layer_idx]
            
            if len(current_inputs) == 0:
                print(f"  Layer {layer_idx:02d}: no quantized inputs collected, using FP32 inputs.")
                current_inputs = layer_inputs_fp[layer_idx]

            n_samples = len(current_inputs)
            n_val = max(1, int(n_samples * val_fraction))
            n_train = n_samples - n_val
            
            shuffled_indices = np.random.permutation(n_samples)
            train_indices = shuffled_indices[:n_train]
            val_indices = shuffled_indices[n_train:]
            
            train_inputs = [current_inputs[i] for i in train_indices]
            val_inputs = [current_inputs[i] for i in val_indices]
            
            print(f"  Layer {layer_idx:02d}: {n_train} train, {n_val} val samples")

            params = [
                q_attn.q_proj.raw_scale,
                q_attn.k_proj.raw_scale,
                q_attn.v_proj.raw_scale,
            ]

            q_attn.force_fp_mode = False

            optimizer = torch.optim.Adam(params, lr=lr)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps_per_layer)

            lam_values = []
            n_lambda_samples = min(8, len(train_inputs))
            if n_lambda_samples > 0:
                sample_indices = np.random.choice(len(train_inputs), n_lambda_samples, replace=False)
                
                for idx in sample_indices:
                    sample_inp = train_inputs[idx].to(device)
                    q_attn.force_fp_mode = True
                    P_fp, O_fp, mask = q_attn.forward_components(sample_inp, use_quant=False)
                    q_attn.force_fp_mode = False
                    P_q, O_q, _ = q_attn.forward_components(sample_inp, use_quant=True)
                    _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=1.0, temp=init_temp
                    )
                    lam_val = float(l_out_0 / (l_kl_0 + 1e-8))
                    lam_values.append(lam_val)
            
            lam_val = min(max(np.mean(lam_values), 0.01), 10.0) if lam_values else 1.0
            print(f"  Initial λ: {lam_val:.4f}")

            temp = init_temp
            best_loss = float("inf")
            best_scales = (
                q_attn.q_proj.raw_scale.data.clone(),
                q_attn.k_proj.raw_scale.data.clone(),
                q_attn.v_proj.raw_scale.data.clone(),
            )

            def sample_batch_loss(inputs, device, q_attn, lam_val, temp, batch_size):
                total_samples = len(inputs)
                if total_samples == 0:
                    return None
                
                n_samples = min(batch_size, total_samples)
                idxs = torch.randint(0, total_samples, (n_samples,))
                
                l_joint_sum = 0.0
                l_out_sum = 0.0
                l_kl_sum = 0.0
                mse_sum = 0.0
                
                for idx in idxs:
                    inp = inputs[idx.item()].to(device)
                    
                    q_attn.force_fp_mode = True
                    with torch.no_grad():
                        P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)
                    
                    q_attn.force_fp_mode = False
                    P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)
                    
                    l_joint, l_out, l_kl, mse_raw = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    
                    l_joint_sum = l_joint_sum + l_joint
                    l_out_sum = l_out_sum + l_out
                    l_kl_sum = l_kl_sum + l_kl
                    mse_sum = mse_sum + mse_raw
                
                return l_joint_sum / n_samples, l_out_sum / n_samples, l_kl_sum / n_samples, mse_sum / n_samples

            def compute_full_val_loss(inputs, device, q_attn, lam_val, temp):
                if len(inputs) == 0:
                    return float('inf')
                
                total_loss = 0.0
                for inp in inputs:
                    inp = inp.to(device)
                    
                    q_attn.force_fp_mode = True
                    with torch.no_grad():
                        P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)
                    
                    q_attn.force_fp_mode = False
                    with torch.no_grad():
                        P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)
                    
                    l_j, _, _, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    total_loss += l_j.item()
                
                return total_loss / len(inputs)

            for step in range(num_steps_per_layer):
                optimizer.zero_grad()

                batch_result = sample_batch_loss(
                    train_inputs, device, q_attn, lam_val, temp, batch_size
                )
                if batch_result is None:
                    continue
                    
                l_joint, l_out, l_kl, mse_raw = batch_result

                l_joint.backward()
                torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)

                optimizer.step()
                scheduler.step()

                if (step + 1) % lambda_update_every == 0:
                    lam_new = float(l_out.item() / (l_kl.item() + 1e-8))
                    lam_new = min(max(lam_new, 0.01), 10.0)
                    lam_val = 0.5 * lam_val + 0.5 * lam_new

                temp = max(1.0, temp * 0.999)

                if (step + 1) % 10 == 0 or step == 0:
                    val_loss = compute_full_val_loss(
                        val_inputs, device, q_attn, lam_val, temp
                    )

                    if val_loss < best_loss:
                        best_loss = val_loss
                        best_scales = (
                            q_attn.q_proj.raw_scale.data.clone(),
                            q_attn.k_proj.raw_scale.data.clone(),
                            q_attn.v_proj.raw_scale.data.clone(),
                        )

                    current_lr = scheduler.get_last_lr()[0]
                    print(f"  Layer {layer_idx:02d} Step [{step+1:03d}/{num_steps_per_layer:03d}] | "
                          f"Train: {l_joint.item():.6f} | Val: {val_loss:.6f} | "
                          f"L_Out: {l_out.item():.6f} | L_KL: {l_kl.item():.6f} | "
                          f"MSE: {mse_raw.item():.6f} | λ: {lam_val:.4f} | Temp: {temp:.3f}")

            q_attn.q_proj.raw_scale.data = best_scales[0]
            q_attn.k_proj.raw_scale.data = best_scales[1]
            q_attn.v_proj.raw_scale.data = best_scales[2]

            q_attn.q_proj.raw_scale.requires_grad = False
            q_attn.k_proj.raw_scale.requires_grad = False
            q_attn.v_proj.raw_scale.requires_grad = False

            print(f"  ✓ Layer {layer_idx:02d} calibrated (best Val Loss={best_loss:.6f})")

        print(f"\n  Pass {pass_idx + 1} completed. Evaluating current model on calibration data...")
        current_ppl = evaluate_ppl(model, calib_tokens, device)
        
        if current_ppl < best_overall_loss:
            best_overall_loss = current_ppl
            best_scales_overall = [(
                q_attn.q_proj.raw_scale.data.clone(),
                q_attn.k_proj.raw_scale.data.clone(),
                q_attn.v_proj.raw_scale.data.clone(),
            ) for q_attn in q_attn_blocks]
            print(f"  ✓ New best PPL: {current_ppl:.2f}")
        else:
            print(f"  Current PPL: {current_ppl:.2f} (best: {best_overall_loss:.2f})")

        if pass_idx < calibration_passes - 1:
            for q_attn in q_attn_blocks:
                q_attn.q_proj.raw_scale.requires_grad = True
                q_attn.k_proj.raw_scale.requires_grad = True
                q_attn.v_proj.raw_scale.requires_grad = True
            print("\n  Unfroze all layers for next calibration pass.")

    print(f"\n  Restoring best scales from pass with PPL {best_overall_loss:.2f}")
    for q_attn, (q_raw, k_raw, v_raw) in zip(q_attn_blocks, best_scales_overall):
        q_attn.q_proj.raw_scale.data = q_raw
        q_attn.k_proj.raw_scale.data = k_raw
        q_attn.v_proj.raw_scale.data = v_raw

    for q_attn in q_attn_blocks:
        q_attn.q_proj.raw_scale.requires_grad = False
        q_attn.k_proj.raw_scale.requires_grad = False
        q_attn.v_proj.raw_scale.requires_grad = False

    print(f"\n✓ Sequential calibration completed with {calibration_passes} passes.")
    print(f"  Best validation PPL: {best_overall_loss:.2f}")
    return model

def calibrate_ap_quant_joint(
    model, 
    calib_tokens, 
    device,
    layer_bits=None,
    num_steps=300, 
    lr=1e-3,
    init_temp=2.0, 
    lambda_update_every=10,
    batch_size=4,
    val_fraction=0.1,
):
    model.eval()
    model.to(device)

    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=layer_bits)

    print("\n" + "=" * 70)
    print(f"AP-QUANT: JOINT CALIBRATION ({arch.upper()}, 4-BIT, DYNAMIC λ, QKV-ONLY + FP32 Wo)")
    print(f"BATCH_SIZE: {batch_size}, VAL_FRACTION: {val_fraction}")
    print("=" * 70)
    
    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    layer_inputs = collect_layer_inputs(model, arch=arch, calib_tokens=calib_tokens, device=device)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    train_inputs = {}
    val_inputs = {}
    
    print("\n  Creating held-out validation sets for each layer...")
    for layer_idx in range(len(q_attn_blocks)):
        n_samples = len(layer_inputs[layer_idx])
        if n_samples == 0:
            train_inputs[layer_idx] = []
            val_inputs[layer_idx] = []
            continue
            
        n_val = max(1, int(n_samples * val_fraction))
        n_train = n_samples - n_val
        
        shuffled_indices = np.random.permutation(n_samples)
        train_indices = shuffled_indices[:n_train]
        val_indices = shuffled_indices[n_train:]
        
        train_inputs[layer_idx] = [layer_inputs[layer_idx][i] for i in train_indices]
        val_inputs[layer_idx] = [layer_inputs[layer_idx][i] for i in val_indices]
        
        print(f"  Layer {layer_idx:02d}: {n_train} train, {n_val} val")
    
    total_train_samples = sum(len(v) for v in train_inputs.values())
    total_val_samples = sum(len(v) for v in val_inputs.values())
    print(f"  Total train samples: {total_train_samples}, val samples: {total_val_samples}")

    params = []
    for q_attn in q_attn_blocks:
        params.append(q_attn.q_proj.raw_scale)
        params.append(q_attn.k_proj.raw_scale)
        params.append(q_attn.v_proj.raw_scale)

    optimizer = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)

    lam_values = []
    valid_layers = [i for i in range(len(q_attn_blocks)) if len(train_inputs[i]) > 0]
    
    if valid_layers:
        for layer_idx in valid_layers[:3]:
            q_attn = q_attn_blocks[layer_idx]
            n_samples = min(8, len(train_inputs[layer_idx]))
            sample_indices = np.random.choice(len(train_inputs[layer_idx]), n_samples, replace=False)
            
            for idx in sample_indices:
                sample_inp = train_inputs[layer_idx][idx].to(device)
                with torch.no_grad():
                    P_fp, O_fp, mask = q_attn.forward_components(sample_inp, use_quant=False)
                    P_q, O_q, _ = q_attn.forward_components(sample_inp, use_quant=True)
                    _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=1.0, temp=init_temp
                    )
                    lam_val = float(l_out_0 / (l_kl_0 + 1e-8))
                    lam_values.append(lam_val)
    
    lam_val = min(max(np.mean(lam_values), 0.01), 10.0) if lam_values else 1.0
    print(f"\n  Initial λ estimate: {lam_val:.4f} (averaged over {len(lam_values)} samples)")

    best_loss = float('inf')
    best_scales = [(
        q_attn.q_proj.raw_scale.data.clone(),
        q_attn.k_proj.raw_scale.data.clone(),
        q_attn.v_proj.raw_scale.data.clone(),
    ) for q_attn in q_attn_blocks]

    temp = init_temp

    def sample_batch_loss(q_attn, layer_inputs_dict, layer_idx, device, lam_val, temp, batch_size):
        total_samples = len(layer_inputs_dict[layer_idx])
        if total_samples == 0:
            return None
        
        n_samples = min(batch_size, total_samples)
        idxs = torch.randint(0, total_samples, (n_samples,))
        
        l_joint_sum = 0.0
        l_out_sum = 0.0
        l_kl_sum = 0.0
        mse_sum = 0.0
        
        for idx in idxs:
            inp = layer_inputs_dict[layer_idx][idx.item()].to(device)
            
            with torch.no_grad():
                P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)
            
            P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)
            
            l_joint, l_out, l_kl, mse_raw = compute_ap_loss_improved(
                P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
            )
            
            l_joint_sum = l_joint_sum + l_joint
            l_out_sum = l_out_sum + l_out
            l_kl_sum = l_kl_sum + l_kl
            mse_sum = mse_sum + mse_raw
        
        return l_joint_sum / n_samples, l_out_sum / n_samples, l_kl_sum / n_samples, mse_sum / n_samples

    for step in range(num_steps):
        optimizer.zero_grad()

        total_l_joint = 0.0
        total_l_out = 0.0
        total_l_kl = 0.0
        total_mse = 0.0
        counted_layers = 0

        for layer_idx, q_attn in enumerate(q_attn_blocks):
            if len(train_inputs[layer_idx]) == 0:
                continue

            batch_result = sample_batch_loss(
                q_attn, train_inputs, layer_idx, device, lam_val, temp, batch_size
            )
            if batch_result is None:
                continue
                
            l_joint, l_out, l_kl, mse_raw = batch_result

            total_l_joint = total_l_joint + l_joint
            total_l_out = total_l_out + l_out
            total_l_kl = total_l_kl + l_kl
            total_mse = total_mse + mse_raw
            counted_layers += 1

        if counted_layers == 0:
            continue

        total_l_joint.backward()

        for q_attn in q_attn_blocks:
            torch.nn.utils.clip_grad_norm_(
                [q_attn.q_proj.raw_scale, q_attn.k_proj.raw_scale,
                q_attn.v_proj.raw_scale],
                max_norm=10.0
            )

        optimizer.step()
        scheduler.step()

        avg_l_out = total_l_out.item() / counted_layers
        avg_l_kl = total_l_kl.item() / counted_layers
        avg_mse = total_mse.item() / counted_layers

        if (step + 1) % lambda_update_every == 0:
            lam_new = avg_l_out / (avg_l_kl + 1e-8)
            lam_new = float(min(max(lam_new, 0.01), 10.0))
            lam_val = 0.5 * lam_val + 0.5 * lam_new

        temp = max(1.0, temp * 0.999)

        with torch.no_grad():
            val_loss = 0.0
            val_count = 0
            for layer_idx, q_attn in enumerate(q_attn_blocks):
                val_samples = val_inputs.get(layer_idx, [])
                for inp in val_samples:
                    inp = inp.to(device)
                    P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False)
                    P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True)
                    l_j, _, _, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    val_loss += l_j.item()
                    val_count += 1
            
            if val_count > 0:
                avg_val_loss = val_loss / val_count
            else:
                avg_val_loss = float('inf')

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            for i, q_attn in enumerate(q_attn_blocks):
                best_scales[i] = (
                    q_attn.q_proj.raw_scale.data.clone(),
                    q_attn.k_proj.raw_scale.data.clone(),
                    q_attn.v_proj.raw_scale.data.clone(),
                )

        if (step + 1) % 10 == 0 or step == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Step [{step+1:03d}/{num_steps:03d}] | "
                  f"Train Loss: {total_l_joint.item():.6f} | "
                  f"Val Loss: {avg_val_loss:.6f} | "
                  f"L_Out(avg): {avg_l_out:.6f} | "
                  f"L_KL(avg): {avg_l_kl:.6f} | "
                  f"MSE(avg): {avg_mse:.6f} | "
                  f"λ: {lam_val:.4f} | Temp: {temp:.3f} | LR: {current_lr:.6f}")

    for q_attn, (q_raw, k_raw, v_raw) in zip(q_attn_blocks, best_scales):
        q_attn.q_proj.raw_scale.data = q_raw
        q_attn.k_proj.raw_scale.data = k_raw
        q_attn.v_proj.raw_scale.data = v_raw

    print(f"\n  Best validation loss: {best_loss:.6f}")

    for q_attn in q_attn_blocks:
        q_attn.q_proj.raw_scale.requires_grad = False
        q_attn.k_proj.raw_scale.requires_grad = False
        q_attn.v_proj.raw_scale.requires_grad = False

    print(f"\n✓ Joint calibration completed for all layers ({arch}, dynamic λ, temp, QKV-ONLY + FP32 Wo).")
    return model

def save_calibration_scales(model, filepath="calibration_scales_joint_dyn.pkl"):
    arch = detect_arch(model)
    scales = {}

    if arch == "gpt2":
        blocks = model.transformer.h
        for layer_idx, block in enumerate(blocks):
            attn = block.attn
            scales[layer_idx] = {
                'q_raw': attn.q_proj.raw_scale.detach().float().cpu().numpy(),
                'k_raw': attn.k_proj.raw_scale.detach().float().cpu().numpy(),
                'v_raw': attn.v_proj.raw_scale.detach().float().cpu().numpy(),
            }
    elif arch == "llama":
        blocks = model.model.layers
        for layer_idx, block in enumerate(blocks):
            attn = block.self_attn
            scales[layer_idx] = {
                'q_raw': attn.q_proj.raw_scale.detach().float().cpu().numpy(),
                'k_raw': attn.k_proj.raw_scale.detach().float().cpu().numpy(),
                'v_raw': attn.v_proj.raw_scale.detach().float().cpu().numpy(),
            }
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    with open(filepath, 'wb') as f:
        pickle.dump(scales, f)
    print(f"✓ Scales saved to {filepath}")

def load_calibration_scales(model, filepath="calibration_scales_joint_dyn.pkl"):
    arch = detect_arch(model)
    with open(filepath, 'rb') as f:
        scales = pickle.load(f)

    if arch == "gpt2":
        blocks = model.transformer.h
        for layer_idx, block in enumerate(blocks):
            if layer_idx in scales:
                attn = block.attn
                attn.q_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['q_raw'], device=attn.q_proj.raw_scale.device
                )
                attn.k_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['k_raw'], device=attn.k_proj.raw_scale.device
                )
                attn.v_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['v_raw'], device=attn.v_proj.raw_scale.device
                )
    elif arch == "llama":
        blocks = model.model.layers
        for layer_idx, block in enumerate(blocks):
            if layer_idx in scales:
                attn = block.self_attn
                attn.q_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['q_raw'], device=attn.q_proj.raw_scale.device
                )
                attn.k_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['k_raw'], device=attn.k_proj.raw_scale.device
                )
                attn.v_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['v_raw'], device=attn.v_proj.raw_scale.device
                )
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    print(f"✓ Scales loaded from {filepath}")
    return model

def evaluate_ppl(model, test_tokens, device):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for chunk in test_tokens:
            input_ids = chunk.to(device)
            outputs = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            num_tokens = input_ids.shape[1] - 1
            total_nll += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

def prepare_tokens(corpus, tokenizer, chunk_size=128, max_len_per_example=256):
    chunks = []
    for text in corpus:
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=max_len_per_example)["input_ids"]
        for i in range(0, enc.size(1), chunk_size):
            chunk = enc[:, i:i+chunk_size]
            if chunk.size(1) >= 2:
                chunks.append(chunk)
    return chunks

def sanity_check_fp_forward(model_orig, model_wrapped, sample_chunk, device, arch):
    model_orig.eval()
    model_wrapped.eval()

    if arch == "gpt2":
        blocks = model_wrapped.transformer.h
        attn_attr = "attn"
    elif arch == "llama":
        blocks = model_wrapped.model.layers
        attn_attr = "self_attn"
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    for block in blocks:
        getattr(block, attn_attr).force_fp_mode = True

    with torch.no_grad():
        input_ids = sample_chunk.to(device)
        out_orig = model_orig(input_ids=input_ids, use_cache=False).logits
        out_wrapped = model_wrapped(input_ids=input_ids, use_cache=False).logits

    for block in blocks:
        getattr(block, attn_attr).force_fp_mode = False

    diff = (out_orig - out_wrapped).abs().max().item()
    print(f"Max logit diff (orig vs wrapped, FP32 path): {diff:.6f}")
    assert diff < 1e-3, "Wrapper's FP32 path doesn't match original — RoPE/GQA bug likely."

model_name = "unsloth/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("\nLoading open-platypus dataset...")
ds = load_dataset("garage-bAInd/Open-Platypus", split="train")

calib_texts = [ds[i]["instruction"] for i in range(256)]
calib_tokens = prepare_tokens(calib_texts, tokenizer, chunk_size=128)

eval_texts = [ds[i]["instruction"] for i in range(256, 512)]
eval_tokens = prepare_tokens(eval_texts, tokenizer, chunk_size=128)

print(f"Calibration chunks: {len(calib_tokens)}")
print(f"Evaluation chunks: {len(eval_tokens)}")

print("\n" + "=" * 70)
print("STEP 1: FP32 BASELINE")
print("=" * 70)

model_fp = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="eager").to(device)

if hasattr(model_fp.config, 'n_layer'):
    n_layer = model_fp.config.n_layer
elif hasattr(model_fp.config, 'num_hidden_layers'):
    n_layer = model_fp.config.num_hidden_layers
else:
    raise ValueError("Unknown model config - cannot determine number of layers")
    
model_fp.eval()
fp_ppl = evaluate_ppl(model_fp, eval_tokens, device)
print(f"FP32 Perplexity: {fp_ppl:.2f}")

print("\n" + "=" * 70)
print("STEP 2: UNOPTIMIZED 4-BIT BASELINE (QKV-ONLY + FP32 Wo)")
print("=" * 70)

model_unopt = AutoModelForCausalLM.from_pretrained(model_name).to(device)
assignments = {i: 4 for i in range(n_layer)}

arch_unopt, _ = quantize_transformer_attn(model_unopt, layer_bits=assignments)
model_unopt.eval()
unopt_ppl = evaluate_ppl(model_unopt, eval_tokens, device)
print(f"Unoptimized 4-bit Perplexity ({arch_unopt}): {unopt_ppl:.2f}")

print("\n" + "=" * 70)
print("STEP 3: SEQUENTIAL AP-QUANT CALIBRATION (DYNAMIC λ, QKV-ONLY + FP32 Wo)")
print("=" * 70)

model_for_sensitivity = AutoModelForCausalLM.from_pretrained(
    model_name, attn_implementation="eager"
).to(device)

candidate_bits = [2, 3, 4, 8, 16]

sensitivity, lam = compute_sensitivity_matrix(
    model=model_for_sensitivity,
    calib_tokens=calib_tokens,
    candidate_bits=candidate_bits,
    temp=2.0,
    device=device,
)
del model_for_sensitivity

target_avg_bits = 4
cost = {b: b for b in candidate_bits}
budget = target_avg_bits * n_layer

assignment, total_loss = solve_mckp(
    sensitivity=sensitivity,
    cost=cost,
    budget=budget,
    candidate_bits=candidate_bits,
)

print("lambda used:", lam)
print("bit assignment:", assignment)

# model_calib = AutoModelForCausalLM.from_pretrained(
#     model_name, attn_implementation="eager"
# ).to(device)

# assignment = {i: 4 for i in range(n_layer)}

# Choose calibration method:
# use_joint = True  # Original joint calibration
# use_sequential = False  # New sequential calibration

# model_calib = calibrate_ap_quant_sequential(
#     model_calib, calib_tokens, device,
#     layer_bits=assignment,
#     num_steps_per_layer=200,  # Increased from 100
#     lr=1e-3,
#     init_temp=2.0,
#     lambda_update_every=10,
#     batch_size=4,  # Now matches joint calibration
#     val_fraction=0.1,
#     calibration_passes=3,  # Two passes for refinement
# )

# # pick a single evaluation chunk for the sanity check
# sample_chunk = eval_tokens[0]
# arch_calib = detect_arch(model_calib)
# sanity_check_fp_forward(model_fp, model_calib, sample_chunk, device, arch_calib)

# save_calibration_scales(model_calib, "calibration_scales_sequential.pkl")

# calib_ppl = evaluate_ppl(model_calib, eval_tokens, device)
# print(f"\nCalibrated 4-bit Perplexity (sequential, dynamic λ, QKV-ONLY + FP32 Wo): {calib_ppl:.2f}")

# print("\n" + "=" * 70)
# print("FINAL SUMMARY")
# print("=" * 70)
# print(f"| {'Model':<20} | {'Perplexity':>12} | {'vs FP32':>10} | {'Recovery':>10} |")
# print(f"|{'-'*22}|{'-'*14}|{'-'*12}|{'-'*12}|")
# print(f"| {'FP32 Reference':<20} | {fp_ppl:>12.2f} | {'0.00':>10} | {'-':>10} |")
# print(f"| {'Unoptimized 4-bit':<20} | {unopt_ppl:>12.2f} | {unopt_ppl - fp_ppl:>+9.2f} | {'0.0%':>10} |")

# if calib_ppl < unopt_ppl:
#     recovery = (unopt_ppl - calib_ppl) / (unopt_ppl - fp_ppl + 1e-8) * 100
#     print(f"| {'AP-Quant Sequential':<20} | {calib_ppl:>12.2f} | {calib_ppl - fp_ppl:>+9.2f} | {recovery:>9.1f}% |")
#     print("=" * 70)
#     print(f"\n Recovery: {recovery:.1f}% of quantization gap recovered!")
#     print(f"   FP32: {fp_ppl:.2f} → Unopt: {unopt_ppl:.2f} → Sequential: {calib_ppl:.2f}")
# else:
#     print(f"| {'AP-Quant Sequential':<20} | {calib_ppl:>12.2f} | {calib_ppl - fp_ppl:>+9.2f} | {'0.0%':>10} |")
#     print("=" * 70)
#     print("\n Sequential calibration did not improve PPL. Tune λ / temp / steps / data size.")

# print("\n" + "=" * 70)
# print("PIPELINE COMPLETE")
# print("=" * 70)

AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED 4-BIT, DYNAMIC λ, QKV-ONLY + FP32 Wo, GPT2+LLaMA)
SEED: 42
Device: cuda


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

...

FP32 Perplexity: 6.73

STEP 2: UNOPTIMIZED 4-BIT BASELINE (QKV-ONLY + FP32 Wo)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

  LLaMA layer 00: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 01: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 02: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 03: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 04: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 05: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 06: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 07: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 08: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 09: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 10: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 11: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 12: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 13: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 14: attention quantized to 4-bit (

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

  LLaMA layer 00: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 01: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 02: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 03: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 04: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 05: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 06: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 07: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 08: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 09: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 10: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 11: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 12: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 13: attention quantized to 4-bit (QKV-ONLY + FP32 Wo)
  LLaMA layer 14: attention quantized to 4-bit (